# Arithmetic Convention

linopy is transitioning to a stricter arithmetic convention for coordinate alignment. This notebook covers:

1. [How to opt in](#how-to-opt-in) to the new behavior
2. [v1 convention](#v1-convention-the-future-default) — strict coordinate matching (the future default)
3. [Legacy convention](#legacy-convention-current-default) — the current default behavior
4. [The `join` parameter](#the-join-parameter) — explicit control over alignment
5. [Migration guide](#migration-guide) — updating your code

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import linopy

## How to opt in

linopy uses a global setting to control arithmetic behavior. The default is `"legacy"` (backward-compatible). To enable the new strict convention:

In [ ]:
linopy.options["arithmetic_convention"] = "v1"

Put this at the top of your script, before any arithmetic. Under `"legacy"`, all legacy codepaths emit a `LinopyDeprecationWarning` to help you find code that needs updating.

To silence the warnings without migrating yet:

```python
import warnings
warnings.filterwarnings('ignore', category=linopy.LinopyDeprecationWarning)
```

**Rollout plan:**
- **Now**: `"legacy"` is the default — nothing breaks.
- **linopy v1**: `"v1"` becomes the default, `"legacy"` is removed.

---

## v1 convention (the future default)

Two rules:

1. **Shared dimensions must match exactly.** When two operands share a dimension, their coordinate labels must be identical. A `ValueError` is raised on mismatch.
2. **Non-shared dimensions broadcast freely.** When dimensions are not shared, operands broadcast over the missing dimensions — for both expressions and constants.

This ensures mismatches never silently produce wrong results, while preserving all standard algebraic laws.

Inspired by [pyoframe](https://github.com/Bravos-Power/pyoframe).

In [ ]:
m = linopy.Model()

time = pd.RangeIndex(5, name="time")
techs = pd.Index(["solar", "wind", "gas"], name="tech")
scenarios = pd.Index(["low", "high"], name="scenario")

x = m.add_variables(lower=0, coords=[time], name="x")
y = m.add_variables(lower=0, coords=[time], name="y")
gen = m.add_variables(lower=0, coords=[time, techs], name="gen")
risk = m.add_variables(lower=0, coords=[techs, scenarios], name="risk")

### What works

In [ ]:
# Same coords — just works
x + y

In [ ]:
# Constant with matching coords
factor = xr.DataArray([2, 3, 4, 5, 6], dims=["time"], coords={"time": time})
x * factor

In [ ]:
# Constant with fewer dims — broadcasts freely
cost = xr.DataArray([1.0, 0.5, 3.0], dims=["tech"], coords={"tech": techs})
gen * cost  # cost broadcasts over time

In [ ]:
# Expression + Expression with non-shared dims — broadcasts freely
gen + risk  # (time, tech) + (tech, scenario) → (time, tech, scenario)

In [ ]:
# Scalar — always fine
x + 5

In [ ]:
# Constraints — RHS with fewer dims broadcasts naturally
capacity = xr.DataArray([100, 80, 50], dims=["tech"], coords={"tech": techs})
m.add_constraints(gen <= capacity, name="cap")  # capacity broadcasts over time

### What raises an error

In [ ]:
y_short = m.add_variables(
    lower=0, coords=[pd.RangeIndex(3, name="time")], name="y_short"
)

try:
    x + y_short  # time coords don't match: [0..4] vs [0..2]
except ValueError as e:
    print("ValueError:", e)

In [ ]:
partial = xr.DataArray([10, 20, 30], dims=["time"], coords={"time": [0, 1, 2]})

try:
    x * partial  # time coords [0..4] vs [0,1,2]
except ValueError as e:
    print("ValueError:", e)

In [ ]:
try:
    x <= partial  # constraint RHS doesn't cover all coords
except ValueError as e:
    print("ValueError:", e)

### NaN propagation

Under v1, NaN values in constants **propagate** through arithmetic — they are not silently replaced with zeros. This makes missing data visible:

In [ ]:
vals = xr.DataArray([1.0, np.nan, 3.0, 4.0, 5.0], dims=["time"], coords={"time": time})
result = x + vals
print("const:", result.const.values)  # NaN at position 1

### Escape hatches

When coordinates don't match or your data contains NaN, you have several options:

**Option 1: `.sel()` — subset before operating**

The cleanest way. Explicitly select matching coordinates:

In [ ]:
x.sel(time=[0, 1, 2]) + y_short

**Option 2: Named methods with `join=`**

All arithmetic operations have named-method equivalents (`.add()`, `.sub()`, `.mul()`, `.div()`, `.le()`, `.ge()`, `.eq()`) that accept a `join` parameter:

In [ ]:
x.add(y_short, join="inner")  # intersection: time [0, 1, 2]

In [ ]:
x.mul(partial, join="left")  # keep x's coords, fill missing with 0

**Option 3: `.assign_coords()` — positional alignment**

When two operands have the same shape but different labels, relabel one to match the other:

In [ ]:
z = m.add_variables(lower=0, coords=[pd.RangeIndex(5, 10, name="time")], name="z")

# z has time=[5..9], x has time=[0..4] — same shape, different labels
x + z.assign_coords(time=x.coords["time"])

**Option 4: `linopy.align()` — multi-operand pre-alignment**

In [ ]:
x_aligned, y_short_aligned = linopy.align(x, y_short, join="outer")
x_aligned + y_short_aligned

**Option 5: `.fillna()` — handle NaN in constants**

Under v1, NaN propagates through arithmetic. If your data has NaN values that represent "no effect" (e.g., missing cost data that should be zero), fill them explicitly before operating:

```python
# Addition/subtraction: fill with 0 (additive identity)
x + data_with_nans.fillna(0)

# Multiplication: fill with 1 to preserve coefficients, or 0 to zero them out
x * scaling_factors.fillna(1)  # NaN means "no scaling"
x * mask.fillna(0)             # NaN means "exclude"

# Division: fill with 1 (multiplicative identity)
x / divisors.fillna(1)
```

In [ ]:
# NaN propagates by default
vals_with_nan = xr.DataArray(
    [1.0, np.nan, 3.0, 4.0, 5.0], dims=["time"], coords={"time": time}
)
print("With NaN:   ", (x + vals_with_nan).const.values)

# Fill explicitly to get legacy-like behavior
print("fillna(0):  ", (x + vals_with_nan.fillna(0)).const.values)

### Algebraic properties

All standard algebraic laws hold under v1. You can freely refactor expressions without worrying about dimension ordering.

| Property | Example |
|---|---|
| **Commutativity of +** | `x + y == y + x` |
| **Commutativity of ×** | `x * c == c * x` |
| **Associativity of +** | `(x + y) + z == x + (y + z)` |
| **Scalar distributivity** | `s * (x + y) == s*x + s*y` |
| **Constant distributivity** | `c[B] * (x[A] + g[A,B]) == c[B]*x[A] + c[B]*g[A,B]` |
| **Additive identity** | `x + 0 == x` |
| **Multiplicative identity** | `x * 1 == x` |
| **Double negation** | `-(-x) == x` |
| **Zero** | `x * 0 == 0` |

**Caveat:** These guarantees only hold for operations involving at least one linopy object. Operations between plain constants (`DataArray + DataArray`) use their library's own rules. To enforce strict matching for xarray operations too:

```python
xr.set_options(arithmetic_join="exact")
```

---

## Legacy convention (current default)

The legacy convention is the current default (`linopy.options["arithmetic_convention"] = "legacy"`). It uses heuristics to handle coordinate mismatches silently. This section describes its behavior for users who haven't migrated yet.

Under legacy, all arithmetic operations emit a `LinopyDeprecationWarning`.

In [ ]:
import warnings

linopy.options["arithmetic_convention"] = "legacy"
warnings.filterwarnings("ignore", category=linopy.LinopyDeprecationWarning)

In [ ]:
m2 = linopy.Model()
time = pd.RangeIndex(5, name="time")
x2 = m2.add_variables(lower=0, coords=[time], name="x")
y2_short = m2.add_variables(
    lower=0, coords=[pd.RangeIndex(3, name="time")], name="y_short"
)

### Size-aware alignment

When two operands share a dimension:
- **Same size**: positional alignment (labels ignored, left operand's labels kept)
- **Different size**: left-join (reindex to the left operand's coordinates, fill with zeros)

In [ ]:
# Different size — left join, fill missing with 0
x2 + y2_short  # y_short drops out at time 3, 4

In [ ]:
# Same size — positional alignment (labels ignored!)
z2 = m2.add_variables(lower=0, coords=[pd.RangeIndex(5, 10, name="time")], name="z")
x2 + z2  # x has time=[0..4], z has time=[5..9], but same size → positional match

### NaN filling

NaN values in constants are silently replaced with operation-specific neutral elements:
- Addition/subtraction: NaN → 0
- Multiplication: NaN → 0 (zeroes out the variable)
- Division: NaN → 1 (no scaling)

In [ ]:
vals = xr.DataArray([1.0, np.nan, 3.0, 4.0, 5.0], dims=["time"], coords={"time": time})
result = x2 + vals
print("const:", result.const.values)  # NaN replaced with 0

### Constraint RHS

In constraints, the RHS is reindexed to the expression's coordinates. Missing positions become NaN, which tells linopy to skip those constraints:

In [ ]:
rhs = xr.DataArray([10, 20, 30], dims=["time"], coords={"time": [0, 1, 2]})
con = x2 <= rhs  # constraint only at time 0, 1, 2; NaN at time 3, 4
con

In [ ]:
# Switch back to v1 for the rest of the notebook
linopy.options["arithmetic_convention"] = "v1"
warnings.resetwarnings()

---

## The `join` parameter

Both conventions support explicit `join=` on named methods. This overrides the default behavior and works identically in both modes.

| `join` | Coordinates kept | Fill behavior |
|--------|-----------------|---------------|
| `"exact"` | Must match | `ValueError` if different |
| `"inner"` | Intersection | No fill needed |
| `"outer"` | Union | Fill with neutral element |
| `"left"` | Left operand's | Fill missing right |
| `"right"` | Right operand's | Fill missing left |
| `"override"` | Left operand's (positional) | Positional alignment |

In [ ]:
m3 = linopy.Model()

i_a = pd.Index([0, 1, 2], name="i")
i_b = pd.Index([1, 2, 3], name="i")

a = m3.add_variables(coords=[i_a], name="a")
b = m3.add_variables(coords=[i_b], name="b")

In [ ]:
# Inner join — intersection (i=1, 2)
a.add(b, join="inner")

In [ ]:
# Outer join — union (i=0, 1, 2, 3)
a.add(b, join="outer")

In [ ]:
# Left join — keep a's coords (i=0, 1, 2)
a.add(b, join="left")

In [ ]:
# Right join — keep b's coords (i=1, 2, 3)
a.add(b, join="right")

In [ ]:
# Override — positional (0↔1, 1↔2, 2↔3), uses a's labels
a.add(b, join="override")

The same `join` parameter works on `.mul()`, `.div()`, `.le()`, `.ge()`, `.eq()`:

In [ ]:
const = xr.DataArray([2, 3, 4], dims=["i"], coords={"i": [1, 2, 3]})

# Multiply, keeping only shared coords
a.mul(const, join="inner")

In [ ]:
# Constraint with left join — only a's coords, NaN at missing RHS positions
rhs = xr.DataArray([10, 20], dims=["i"], coords={"i": [0, 1]})
a.le(rhs, join="left")

---

## Migration guide

To migrate from legacy to v1:

### Step 1: Enable v1 and run your code

```python
linopy.options["arithmetic_convention"] = "v1"
```

Any code that relied on legacy alignment will now raise `ValueError` with a helpful message suggesting which `join=` to use.

### Step 2: Fix each error

Common patterns:

| Legacy code (silent) | v1 equivalent (explicit) |
|---|---|
| `x + subset_constant` | `x.add(subset_constant, join="left")` |
| `x + y` (same size, different labels) | `x + y.assign_coords(time=x.coords["time"])` |
| `x <= partial_rhs` | `x.le(partial_rhs, join="left")` |
| `expr + expr` (mismatched coords) | `expr.add(other, join="outer")` or `.sel()` first |

### Step 3: Handle NaN

Under legacy, NaN in constants was silently replaced with 0. Under v1, NaN propagates. If your data contains NaN that should be treated as zero, use `.fillna(0)` explicitly:

```python
# Legacy: NaN silently became 0
x + data_with_nans

# v1: be explicit
x + data_with_nans.fillna(0)
```

### Step 4: Pandas index names

Under v1, pandas objects must have **named indices** to align properly with linopy variables:

```python
# Will fail — unnamed index becomes "dim_0"
cost = pd.Series([10, 20], index=["wind", "solar"])

# Works — explicit dimension name
cost = pd.Series([10, 20], index=pd.Index(["wind", "solar"], name="tech"))
```

---

## Practical example

A generation dispatch model demonstrating both matching coords and explicit joins.

In [ ]:
m4 = linopy.Model()

hours = pd.RangeIndex(24, name="hour")
techs = pd.Index(["solar", "wind", "gas"], name="tech")

gen = m4.add_variables(lower=0, coords=[hours, techs], name="gen")

In [ ]:
# Capacity limits — constant broadcasts over hours
capacity = xr.DataArray([100, 80, 50], dims=["tech"], coords={"tech": techs})
m4.add_constraints(gen <= capacity, name="capacity_limit")

In [ ]:
# Solar availability — full 24h profile, matching coords
solar_avail = np.zeros(24)
solar_avail[6:19] = 100 * np.sin(np.linspace(0, np.pi, 13))
solar_availability = xr.DataArray(solar_avail, dims=["hour"], coords={"hour": hours})

solar_gen = gen.sel(tech="solar")
m4.add_constraints(solar_gen <= solar_availability, name="solar_avail")

In [ ]:
# Peak demand — only applies to hours 8-20, use join="inner"
peak_hours = pd.RangeIndex(8, 21, name="hour")
peak_demand = xr.DataArray(
    np.full(len(peak_hours), 120.0), dims=["hour"], coords={"hour": peak_hours}
)

total_gen = gen.sum("tech")
m4.add_constraints(total_gen.ge(peak_demand, join="inner"), name="peak_demand")

---

## Summary

| | v1 (future default) | Legacy (current default) |
|---|---|---|
| **Mismatched coords** | `ValueError` | Silent left-join / override |
| **Same-size different labels** | `ValueError` | Positional alignment |
| **NaN in constants** | Propagates | Filled with 0 |
| **Explicit join** | `.add(x, join=...)` | `.add(x, join=...)` |
| **Setting** | `options["arithmetic_convention"] = "v1"` | `options["arithmetic_convention"] = "legacy"` |